In [1]:
import os
import numpy as np
import pandas as pd
from scipy.optimize import minimize

PRICES_DIR    = "klines csv data/prices_cleaned"
XGB_BASE_PATH = "Results New/XGBoost Base Forecast new return.npy"
XGB_FEAT_PATH = "Results New/XGBoost Features Forecast new return.npy"
XGB_BO_PATH   = "Results New (1st)/Features and BO new return.npy"
ARIMA_PATH    = "Results New/ARIMA_test_7d_rebalance_forecast_matrix.npy"
LSTM_PATH     = "Results New/lstm_cov_matrices.npy"
DCC_PATH      = "Results New/dcc_garch_daily_covariance_matrices_test.npy"
PERF_DIR      = "15 CMVO results newer"

os.makedirs(PERF_DIR, exist_ok=True)

In [2]:
# 1. LOAD PRICE DATA
all_coins = sorted(os.listdir(PRICES_DIR))
all_coins = [c for c in all_coins if not c.endswith('.keras')
             and not c.endswith('.npy') and not c.endswith('.csv')]

glimpse  = pd.read_csv(PRICES_DIR + "/" + all_coins[0])
combined = pd.DataFrame({
    'date': pd.date_range(start='2022-04-01', periods=len(glimpse), freq='D')
})
for coin in all_coins:
    combined[coin] = pd.read_csv(PRICES_DIR + "/" + coin).sort_values(by='time')['close'].values

combined    = combined.set_index('date')
total_rows  = len(combined)
test_start  = int(total_rows * 0.8)
test_prices = combined.iloc[test_start:].copy()

print(f"Price data shape: {combined.shape}")
print(f"Test period: {test_prices.index[0].date()} to {test_prices.index[-1].date()}")

Price data shape: (1371, 8)
Test period: 2025-04-01 to 2025-12-31


In [3]:
# 2. LOAD XGBOOST PREDICTIONS
def parse_xgb(path, all_coins):
    xgb_raw = np.load(path, allow_pickle=True)
    df = pd.DataFrame(xgb_raw)
    if df.shape[1] == 1:
        df = df[0].astype(str).str.split(",", expand=True)
    df = df.iloc[:, 0:4].copy()
    df.columns = ['coin', 'actual', 'forecast', 'returns_predicted']
    df['coin'] = df['coin'].astype(str).str.strip()
    df = df[df['coin'].str.lower() != 'coin'].reset_index(drop=True)
    df['returns_predicted'] = pd.to_numeric(df['returns_predicted'], errors='coerce')
    mu_list = []
    for coin in all_coins:
        coin_data = df[df['coin'] == str(coin).strip()]['returns_predicted'].values
        coin_data = coin_data[~np.isnan(coin_data)]
        mu_list.append(coin_data)
    non_empty = [x for x in mu_list if len(x) > 0]
    if len(non_empty) == 0:
        return np.array([])
    min_len = min(len(x) for x in non_empty)
    return np.column_stack([x[:min_len] for x in non_empty])

mu_base     = parse_xgb(XGB_BASE_PATH, all_coins)
mu_features = parse_xgb(XGB_FEAT_PATH, all_coins)
mu_bo       = parse_xgb(XGB_BO_PATH,   all_coins)
print(f"mu_base shape: {mu_base.shape}")
print(f"mu_features shape: {mu_features.shape}")
print(f"mu_bo shape: {mu_bo.shape}")

mu_base shape: (39, 8)
mu_features shape: (37, 8)
mu_bo shape: (39, 8)


In [4]:
# 3. LOAD COVARIANCE MATRICES AND ALIGN
REBAL_DAYS = 7

lstm_full = np.load(LSTM_PATH, allow_pickle=True)
lstm_test = lstm_full[test_start:]
lstm_cov  = lstm_test[::REBAL_DAYS]

dcc_full = np.load(DCC_PATH, allow_pickle=True)
dcc_cov  = dcc_full[::REBAL_DAYS]

arima_mu = np.load(ARIMA_PATH)

test_rows       = len(test_prices)
expected_rebals = (test_rows - 1) // REBAL_DAYS
num_rebal = min(
    mu_base.shape[0], mu_features.shape[0], mu_bo.shape[0],
    lstm_cov.shape[0], dcc_cov.shape[0], expected_rebals,
)

mu_base     = mu_base[:num_rebal]
mu_features = mu_features[:num_rebal]
mu_bo       = mu_bo[:num_rebal]
lstm_cov    = lstm_cov[:num_rebal]
dcc_cov     = dcc_cov[:num_rebal]
arima_mu    = arima_mu[:num_rebal]

print(f"Aligned rebalances: {num_rebal}")

Aligned rebalances: 35


## Three solvers: (0.0, 0.3) vs (0.0, 0.4) vs (0.0, 0.5)

Fix from previous notebooks: bounds were `(-0.1, 0.3)` but weights were clipped to `(0.0, 0.3)` after solving — the optimizer was finding solutions under the wrong constraints. Now bounds and clip are consistent.

In [5]:
# 4. SOLVERS — long-only, bounds match what we actually trade
def make_solver(upper_bound):
    def solve_cmvo(mu, cov, risk_aversion=1.0):
        n       = len(mu)
        cov_reg = cov + 1e-8 * np.eye(n)

        mu_scale  = np.abs(mu).mean()
        cov_scale = np.diag(cov_reg).mean()
        scale     = mu_scale / cov_scale if cov_scale > 0 else 1.0
        cov_scaled = cov_reg * scale

        def neg_obj(w):
            return -(w @ mu - (risk_aversion / 2.0) * (w @ cov_scaled @ w))

        res = minimize(
            neg_obj,
            x0          = np.full(n, 1.0 / n),
            method      = 'SLSQP',
            bounds      = [(0.0, upper_bound)] * n,
            constraints = [{'type': 'eq', 'fun': lambda w: w.sum() - 1.0}],
            options     = {'ftol': 1e-9, 'maxiter': 1000},
        )

        if res.success:
            w  = np.clip(res.x, 0.0, upper_bound)
            w /= w.sum()
            return w
        return np.full(n, 1.0 / n)
    return solve_cmvo

solve_03 = make_solver(0.3)
solve_04 = make_solver(0.4)
solve_05 = make_solver(0.5)

In [6]:
# 5. PORTFOLIO SIMULATION
def run_portfolio(test_prices, all_coins, mu_array, cov_array, solver, cost=0.001):
    prices     = test_prices[all_coins].values
    n          = len(all_coins)
    REBAL_DAYS = 7

    holdings         = (1.0 / n) / prices[0]
    portfolio_values = [1.0]
    pct_change       = [0.0]
    rebal_count      = 0

    for i in range(1, len(prices)):
        current_value = (holdings * prices[i]).sum()

        if i % REBAL_DAYS == 0:
            if rebal_count < len(mu_array):
                new_weights = solver(mu_array[rebal_count], cov_array[rebal_count])
                rebal_count += 1
            else:
                new_weights = np.full(n, 1.0 / n)

            tc            = (np.abs(holdings * prices[i] - new_weights * current_value) * cost).sum()
            net_value     = current_value - tc
            holdings      = (new_weights * net_value) / prices[i]
            current_value = net_value

        portfolio_values.append(current_value)
        pct_change.append((current_value / portfolio_values[-2]) - 1.0)

    return pd.DataFrame({'portfolio_values': portfolio_values, 'pct_change': pct_change},
                        index=test_prices.index)

In [7]:
# 6. METRICS
def calculate_metrics(result):
    pct_changes      = result['pct_change'].iloc[1:]
    portfolio_values = result['portfolio_values']

    cumulative   = (1 + pct_changes).cumprod()
    rolling_peak = cumulative.cummax()
    drawdown     = (cumulative - rolling_peak) / rolling_peak
    mdd          = drawdown.min()

    mean   = pct_changes.mean()
    std    = pct_changes.std()
    sharpe = (mean / std) if std != 0 else 0.0

    total_return = (portfolio_values.iloc[-1] / portfolio_values.iloc[0]) - 1
    calmar       = total_return / abs(mdd) if mdd != 0 else 0.0

    downside = pct_changes[pct_changes < 0]
    down_std = downside.std()
    sortino  = (mean / down_std) if down_std != 0 else 0.0

    return {
        'final_value':  portfolio_values.iloc[-1],
        'total_return': total_return,
        'mdd':          mdd,
        'sharpe':       sharpe,
        'sortino':      sortino,
        'calmar':       calmar,
    }

In [8]:
# 7. RUN BENCHMARKS
def run_equal_weight(test_prices, all_coins):
    n        = len(all_coins)
    prices   = test_prices[all_coins].values
    holdings = (1.0 / n) / prices[0]
    pv, pc   = [1.0], [0.0]
    for i in range(1, len(prices)):
        cv = (holdings * prices[i]).sum()
        pv.append(cv)
        pc.append((cv / pv[-2]) - 1.0)
    return pd.DataFrame({'portfolio_values': pv, 'pct_change': pc})

def run_mv_benchmark(test_prices, all_coins, lookback=60, cost=0.001):
    n               = len(all_coins)
    prices          = test_prices[all_coins].values
    current_weights = np.full(n, 1.0 / n)
    holdings        = current_weights / prices[0]
    pv, pc          = [1.0], [0.0]
    REBAL_DAYS      = 7
    for i in range(1, len(prices)):
        cv = (holdings * prices[i]).sum()
        if i % REBAL_DAYS == 0:
            start          = max(0, i - lookback)
            wr             = np.diff(prices[start:i], axis=0) / prices[start:i][:-1]
            if len(wr) >= 2:
                mu_h  = wr.mean(axis=0)
                cov_h = np.cov(wr.T) + 1e-8 * np.eye(n)
                current_weights = solve_03(mu_h, cov_h)
            else:
                current_weights = np.full(n, 1.0 / n)
            tc       = (np.abs(holdings * prices[i] - current_weights * cv) * cost).sum()
            net      = cv - tc
            holdings = (current_weights * net) / prices[i]
            cv       = net
        pv.append(cv)
        pc.append((cv / pv[-2]) - 1.0)
    return pd.DataFrame({'portfolio_values': pv, 'pct_change': pc})

m_1n = calculate_metrics(run_equal_weight(test_prices, all_coins))
m_mv = calculate_metrics(run_mv_benchmark(test_prices, all_coins))
print("Benchmarks computed.")

Benchmarks computed.


In [9]:
# 8. RUN ALL MODELS WITH ALL BOUND VARIANTS
MODEL_CONFIGS = [
    ('xgb_base_lstm',     mu_base,     lstm_cov),
    ('xgb_features_lstm', mu_features, lstm_cov),
    ('xgb_bo_lstm',       mu_bo,       lstm_cov),
    ('xgb_base_dcc',      mu_base,     dcc_cov),
    ('xgb_features_dcc',  mu_features, dcc_cov),
    ('xgb_bo_dcc',        mu_bo,       dcc_cov),
    ('arima_lstm',        arima_mu,    lstm_cov),
    ('arima_dcc',         arima_mu,    dcc_cov),
]

results_03 = {}
results_04 = {}
results_05 = {}

for model_name, mu_array, cov in MODEL_CONFIGS:
    res03 = run_portfolio(test_prices, all_coins, mu_array, cov, solve_03)
    res04 = run_portfolio(test_prices, all_coins, mu_array, cov, solve_04)
    res05 = run_portfolio(test_prices, all_coins, mu_array, cov, solve_05)
    results_03[model_name] = calculate_metrics(res03)
    results_04[model_name] = calculate_metrics(res04)
    results_05[model_name] = calculate_metrics(res05)
    res03.to_csv(os.path.join(PERF_DIR, f'{model_name}_03.csv'), index=True)
    res04.to_csv(os.path.join(PERF_DIR, f'{model_name}_04.csv'), index=True)
    res05.to_csv(os.path.join(PERF_DIR, f'{model_name}_05.csv'), index=True)

print("Done.")

Done.


In [10]:
# 9. COMPARISON TABLE
HDR = f"{'Model':<25} {'Final':>8} {'Return':>8} {'Sharpe':>8} {'MDD':>8} {'Sortino':>8} {'Calmar':>8}"
SEP = "-" * 80

def print_row(name, m):
    print(f"{name:<25} {m['final_value']:>8.4f} {m['total_return']:>8.4f} "
          f"{m['sharpe']:>8.4f} {m['mdd']:>8.4f} {m['sortino']:>8.4f} {m['calmar']:>8.4f}")

for label, results in [('BOUNDS (0.0, 0.3)', results_03), ('BOUNDS (0.0, 0.4)', results_04), ('BOUNDS (0.0, 0.5)', results_05)]:
    print(f"\n=== {label} (test period: Apr-Dec 2025, cost=0.001) ===")
    print(HDR)
    print(SEP)
    print_row('1/N Equal Weight', m_1n)
    print_row('Statistical MV',   m_mv)
    print(SEP)
    for name, m in results.items():
        print_row(name, m)


=== BOUNDS (0.0, 0.3) (test period: Apr-Dec 2025, cost=0.001) ===
Model                        Final   Return   Sharpe      MDD  Sortino   Calmar
--------------------------------------------------------------------------------
1/N Equal Weight            1.1855   0.1855   0.0363  -0.3377   0.0545   0.5491
Statistical MV              1.2646   0.2646   0.0452  -0.3176   0.0657   0.8330
--------------------------------------------------------------------------------
xgb_base_lstm               1.0477   0.0477   0.0200  -0.3703   0.0274   0.1287
xgb_features_lstm           1.0463   0.0463   0.0195  -0.3165   0.0323   0.1464
xgb_bo_lstm                 1.0163   0.0163   0.0177  -0.4485   0.0267   0.0363
xgb_base_dcc                1.0392   0.0392   0.0188  -0.3635   0.0262   0.1079
xgb_features_dcc            1.0419   0.0419   0.0189  -0.2888   0.0308   0.1452
xgb_bo_dcc                  1.0805   0.0805   0.0248  -0.4192   0.0376   0.1920
arima_lstm                  0.9090  -0.0910  -0.001